# Huấn Luyện & Đánh Giá Mô Hình CKAN Trên GPU (Google Colab)
### DSS Movie Recommendation System: Collaborative Knowledge-aware Attentive Network

Notebook này phục vụ việc huấn luyện mô hình **CKAN (Collaborative Knowledge-aware Attentive Network)** dựa trên Đồ thị Tri thức (Knowledge Graph) cho hệ thống gợi ý phim:
1. **Chuẩn bị dữ liệu**: Tải bộ dữ liệu tương tác người dùng (MovieLens) và Đồ thị Tri thức phim (Satori KG).
2. **Lan truyền tri thức (KG Propagation)**: Khởi tạo các ripple set để thu thập ngữ cảnh ngữ nghĩa (Đạo diễn, Diễn viên, Thể loại).
3. **Huấn luyện mô hình**: Học biểu diễn Attention tri thức trên GPU T4 và tự động lưu checkpoint xuất sắc nhất (`ckan_model.pt`).
4. **Đánh giá chất lượng**: Trực quan hóa đường cong hàm mất mát (Loss), chỉ số ROC-AUC và F1-Score trên tập kiểm thử.
5. **Xuất mô hình**: Tải file checkpoint `ckan_model.pt` tương thích 100% về máy tính để nạp trực tiếp vào Backend FastAPI.

> **Lưu ý Colab**: Hãy đảm bảo bạn đã bật GPU: **Runtime (Thời gian chạy) -> Change runtime type (Thay đổi loại phần cứng) -> Chọn T4 GPU**.


In [ ]:
# ============================================================
# 1. KIỂM TRA MÔI TRƯỜNG & GPU
# ============================================================
import torch

print("=== KIỂM TRA PHẦN CỨNG ===")
print("Phiên bản PyTorch :", torch.__version__)
print("CUDA khả dụng     :", torch.cuda.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Tên GPU           :", torch.cuda.get_device_name(0))
    print(f"Bộ nhớ GPU        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = torch.device("cpu")
    print("[Cảnh báo] Đang dùng CPU. Khuyên bạn nên đổi sang GPU T4 trong Runtime -> Change runtime type.")



In [ ]:
# ============================================================
# 2. CÀI ĐẶT THƯ VIỆN BỔ TRỢ
# ============================================================
!pip install -q --upgrade scikit-learn matplotlib seaborn pandas tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, f1_score
from collections import defaultdict
import os
import shutil
import urllib.request
import zipfile

print("[OK] Đã nạp thành công các thư viện cần thiết!")



In [ ]:
# ============================================================
# 3. THIẾT LẬP SIÊU THAM SỐ (HYPERPARAMETERS)
# ============================================================
DATASET     = "movie"   # Mặc định bộ phim (MovieLens + KG Satori)
DIM         = 64        # Chiều vector biểu diễn Embedding
N_LAYER     = 1         # Số bước lan truyền tri thức (L=1 tối ưu cho MovieLens)
UTSS        = 32        # Kích thước tập bộ ba user (User Triple Set Size)
ITSS        = 32        # Kích thước tập bộ ba item (Item Triple Set Size)
AGG         = "sum"     # Hàm tổng hợp đặc trưng ('sum', 'concat', 'neighbor')
BATCH_SIZE  = 1024      # Kích thước batch tối ưu cho GPU T4
N_EPOCH     = 10        # Số vòng huấn luyện
LEARNING_RATE = 0.002   # Tốc độ học (Adam Optimizer)
L2_WEIGHT   = 1e-5      # Trọng số chuẩn hóa L2 chống overfitting

print(f"Cấu hình: Dataset={DATASET}, Dim={DIM}, N_Layer={N_LAYER}, Batch={BATCH_SIZE}, Epochs={N_EPOCH}, LR={LEARNING_RATE}")



## 4. Tải Dữ Liệu Huấn Luyện (Fast-track)
Tải trực tiếp ma trận tương tác (`ratings_final.npy`) và đồ thị tri thức (`kg_final.npy`) đã tiền xử lý chuẩn mực.


In [ ]:
# ============================================================
# 4. TẢI DỮ LIỆU TỐC ĐỘ CAO (FAST-TRACK)
# ============================================================
DATA_DIR = "./data/movie"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("./models", exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/Chinh-de/dss_ckan/main/backend/data/movie"
DATA_FILES = ["ratings_final.npy", "kg_final.npy"]

def download_file(url, dest_path):
    if not os.path.exists(dest_path):
        print(f"Đang tải: {os.path.basename(dest_path)}...")
        try:
            urllib.request.urlretrieve(url, dest_path)
            print(f"Đã tải xong: {dest_path}")
        except Exception as e:
            print(f"Không thể tải từ GitHub: {e}. Thử tải từ nguồn dự phòng...")
            alt_url = f"https://github.com/Chinh-de/dss_ckan/raw/main/backend/data/movie/{os.path.basename(dest_path)}"
            urllib.request.urlretrieve(alt_url, dest_path)
            print(f"Đã tải xong từ nguồn dự phòng: {dest_path}")
    else:
        print(f"Đã có sẵn: {dest_path}")

for fname in DATA_FILES:
    dest = os.path.join(DATA_DIR, fname)
    download_file(f"{BASE_URL}/{fname}", dest)

# Nạp ma trận dữ liệu
rating_np = np.load(f"{DATA_DIR}/ratings_final.npy")
kg_np = np.load(f"{DATA_DIR}/kg_final.npy")

n_entity = int(max(np.max(kg_np[:, 0]), np.max(kg_np[:, 2]))) + 1
n_relation = int(np.max(kg_np[:, 1])) + 1
n_user = int(np.max(rating_np[:, 0])) + 1
n_item = int(np.max(rating_np[:, 1])) + 1

print(f"\n--- THỐNG KÊ DỮ LIỆU ---")
print(f"  - Số lượng User        : {n_user:,}")
print(f"  - Số lượng Item (Phim) : {n_item:,}")
print(f"  - Số bản ghi tương tác : {len(rating_np):,}")
print(f"  - Số Entity (KG)       : {n_entity:,}")
print(f"  - Số Relation (KG)     : {n_relation:,}")
print(f"  - Số bộ ba Tri thức    : {len(kg_np):,}")



## 5. Phân Chia Dữ Liệu (6:2:2) & Khởi Tạo Lan Truyền Tri Thức


In [ ]:
# ============================================================
# 5. CHIA TẬP TRAIN / EVAL / TEST (6:2:2) & KG PROPAGATION
# ============================================================
np.random.seed(42)

def dataset_split(rating_np):
    n_rows = rating_np.shape[0]
    idx = np.random.permutation(n_rows)
    eval_end = int(n_rows * 0.6)
    test_end = int(n_rows * 0.8)

    train_data = rating_np[idx[:eval_end]]
    eval_data  = rating_np[idx[eval_end:test_end]]
    test_data  = rating_np[idx[test_end:]]

    user_seed, item_seed = defaultdict(list), defaultdict(list)
    for u, i, r in train_data:
        if r == 1:
            user_seed[u].append(i)
            item_seed[i].append(u)
    return train_data, eval_data, test_data, user_seed, item_seed

def construct_kg(kg_np):
    kg = defaultdict(list)
    for h, r, t in kg_np:
        kg[h].append((h, r, t))
    return kg

def kg_propagation(kg, seed_dict, set_size, n_layer):
    triple_sets = {}
    for obj, seeds in seed_dict.items():
        layers = []
        if len(seeds) == 0:
            layers.append(([0] * set_size, [0] * set_size, [0] * set_size))
        else:
            idx = np.random.choice(len(seeds), size=set_size, replace=(len(seeds) < set_size))
            layers.append(([seeds[i] for i in idx], [0] * set_size, [0] * set_size))

        for l in range(n_layer):
            h_prev = layers[-1][0] if l == 0 else layers[-1][2]
            h_list, r_list, t_list = [], [], []
            for h in h_prev:
                triples = kg.get(h, [])
                if len(triples) == 0:
                    continue
                chosen_idx = np.random.choice(len(triples))
                chosen = triples[chosen_idx]
                h_list.append(chosen[0])
                r_list.append(chosen[1])
                t_list.append(chosen[2])
            
            if len(h_list) == 0:
                layers.append(layers[-1] if l > 0 else ([0]*set_size, [0]*set_size, [0]*set_size))
            else:
                idx = np.random.choice(len(h_list), size=set_size, replace=(len(h_list) < set_size))
                layers.append(([h_list[i] for i in idx], [r_list[i] for i in idx], [t_list[i] for i in idx]))
        triple_sets[obj] = layers
    return triple_sets

print("Đang chia dữ liệu và xây dựng mạng lan truyền tri thức...")
train_data, eval_data, test_data, user_seed, item_seed = dataset_split(rating_np)
kg = construct_kg(kg_np)
user_triple_set = kg_propagation(kg, user_seed, UTSS, N_LAYER)
item_triple_set = kg_propagation(kg, item_seed, ITSS, N_LAYER)

print(f"  - Train samples : {train_data.shape[0]:,}")
print(f"  - Eval samples  : {eval_data.shape[0]:,}")
print(f"  - Test samples  : {test_data.shape[0]:,}")
print("[OK] Dữ liệu chia và lan truyền tri thức hoàn tất!")



## 6. Định Nghĩa Cấu Trúc Mô Hình CKAN
Kiến trúc mạng nơ-ron **CKAN (Collaborative Knowledge-aware Attentive Network)**:
- Không lưu embedding tĩnh của User; vector người dùng được sinh động từ tương tác và tri thức mở rộng.
- Cơ chế Attention đa tầng đánh trọng số ngữ cảnh giữa các liên kết quan hệ (Relationship) và thực thể (Entity).
- Hàm tổng hợp kết hợp các tầng biểu diễn để tính xác suất người dùng tương tác với phim.


In [ ]:
# ============================================================
# 6. ĐỊNH NGHĨA MÔ HÌNH CKAN
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class CKAN(nn.Module):
    def __init__(self, n_entity, n_relation, dim, n_layer, agg="sum"):
        super(CKAN, self).__init__()
        self.n_entity = n_entity
        self.n_relation = n_relation
        self.dim = dim
        self.n_layer = n_layer
        self.agg = agg

        self.entity_emb = nn.Embedding(self.n_entity, self.dim)
        self.relation_emb = nn.Embedding(self.n_relation, self.dim)
        self.attention = nn.Sequential(
            nn.Linear(self.dim * 4, self.dim),
            nn.ReLU(),
            nn.Linear(self.dim, 1)
        )

    def _attention(self, h_emb, r_emb, t_emb, v):
        # h, r, t: [batch_size, triple_set_size, dim]
        # v: [batch_size, dim]
        v_expanded = v.unsqueeze(1).expand_as(h_emb)
        att_input = torch.cat([h_emb, r_emb, t_emb, v_expanded], dim=-1)
        att_weights = self.attention(att_input)  # [batch_size, triple_set_size, 1]
        att_weights = F.softmax(att_weights, dim=1)
        res = (att_weights * t_emb).sum(dim=1)   # [batch_size, dim]
        return res

    def forward(self, items, user_triple_set, item_triple_set):
        # Embedding của item mục tiêu
        item_emb = self.entity_emb(items)  # [batch_size, dim]

        # 1. Lan truyền tri thức phía User (User Ripple Set Attention)
        user_embs = []
        for l in range(self.n_layer):
            h_idx, r_idx, t_idx = user_triple_set[0][l], user_triple_set[1][l], user_triple_set[2][l]
            h = self.entity_emb(h_idx)
            r = self.relation_emb(r_idx)
            t = self.entity_emb(t_idx)
            user_rep = self._attention(h, r, t, item_emb)
            user_embs.append(user_rep)

        # 2. Lan truyền tri thức phía Item (Item Ripple Set Attention)
        item_embs = [item_emb]
        for l in range(self.n_layer):
            h_idx, r_idx, t_idx = item_triple_set[0][l], item_triple_set[1][l], item_triple_set[2][l]
            h = self.entity_emb(h_idx)
            r = self.relation_emb(r_idx)
            t = self.entity_emb(t_idx)
            item_rep = self._attention(h, r, t, user_embs[0])
            item_embs.append(item_rep)

        # 3. Hàm tổng hợp (Aggregator)
        if self.agg == "sum":
            e_u = sum(user_embs)
            e_v = sum(item_embs)
        elif self.agg == "neighbor":
            e_u = user_embs[-1]
            e_v = item_embs[-1]
        else:
            e_u = user_embs[0]
            e_v = item_embs[0]
            for i in range(1, len(user_embs)):
                e_u = e_u + user_embs[i]
            for i in range(1, len(item_embs)):
                e_v = e_v + item_embs[i]

        return torch.sigmoid((e_v * e_u).sum(dim=-1))

print("[OK] Đã định nghĩa xong cấu trúc mô hình CKAN!")



## 7. Huấn Luyện Mô Hình CKAN
Huấn luyện mô hình CKAN với mạng Attention tri thức và tự động lưu file checkpoint tốt nhất (`ckan_model.pt`).


In [ ]:
# ============================================================
# 7. HUẤN LUYỆN CKAN (WITH KNOWLEDGE GRAPH)
# ============================================================
def to_triple_tensor(objs, triple_set, n_layer, dev):
    h, r, t = [], [], []
    for l in range(n_layer):
        h.append(torch.LongTensor([triple_set.get(o, triple_set[list(triple_set.keys())[0]])[l][0] for o in objs]).to(dev))
        r.append(torch.LongTensor([triple_set.get(o, triple_set[list(triple_set.keys())[0]])[l][1] for o in objs]).to(dev))
        t.append(torch.LongTensor([triple_set.get(o, triple_set[list(triple_set.keys())[0]])[l][2] for o in objs]).to(dev))
    return [h, r, t]

def evaluate_ckan(model, data, b_size=BATCH_SIZE):
    model.eval()
    aucs, f1s = [], []
    with torch.no_grad():
        for start in range(0, data.shape[0], b_size):
            end = min(start + b_size, data.shape[0])
            batch = data[start:end]
            items = torch.LongTensor(batch[:, 1]).to(device)
            u_tr = to_triple_tensor(batch[:, 0].tolist(), user_triple_set, N_LAYER, device)
            i_tr = to_triple_tensor(batch[:, 1].tolist(), item_triple_set, N_LAYER, device)
            scores = model(items, u_tr, i_tr).cpu().numpy()
            labels = batch[:, 2]
            if len(np.unique(labels)) > 1:
                aucs.append(roc_auc_score(labels, scores))
                f1s.append(f1_score(labels, (scores >= 0.5).astype(int), zero_division=0))
    model.train()
    return float(np.mean(aucs)), float(np.mean(f1s))

# Khởi tạo mô hình CKAN
ckan_model = CKAN(n_entity, n_relation, DIM, N_LAYER, AGG).to(device)
opt_ckan = torch.optim.Adam(ckan_model.parameters(), lr=LEARNING_RATE, weight_decay=L2_WEIGHT)
bce_loss = nn.BCELoss()

ckan_history = {
    "epoch": [],
    "loss": [],
    "eval_auc": [],
    "eval_f1": [],
    "test_auc": [],
    "test_f1": []
}

best_ckan_auc = 0.0
BEST_CKAN_PATH = "./models/ckan_model.pt"

print(f"=== BẮT ĐẦU HUẤN LUYỆN CKAN ({N_EPOCH} EPOCHS) ===")
print("-" * 65)
print(f"{'Epoch':<6} | {'Train Loss':<12} | {'Eval AUC':<10} {'Eval F1':<10} | {'Test AUC':<10} {'Test F1':<10}")
print("-" * 65)

for epoch in range(1, N_EPOCH + 1):
    np.random.shuffle(train_data)
    losses = []

    for start in range(0, train_data.shape[0], BATCH_SIZE):
        end = min(start + BATCH_SIZE, train_data.shape[0])
        batch = train_data[start:end]
        items = torch.LongTensor(batch[:, 1]).to(device)
        labels = torch.FloatTensor(batch[:, 2]).to(device)
        u_tr = to_triple_tensor(batch[:, 0].tolist(), user_triple_set, N_LAYER, device)
        i_tr = to_triple_tensor(batch[:, 1].tolist(), item_triple_set, N_LAYER, device)

        opt_ckan.zero_grad()
        preds = ckan_model(items, u_tr, i_tr)
        loss = bce_loss(preds, labels)
        loss.backward()
        opt_ckan.step()
        losses.append(loss.item())

    ev_auc, ev_f1 = evaluate_ckan(ckan_model, eval_data)
    te_auc, te_f1 = evaluate_ckan(ckan_model, test_data)

    ckan_history["epoch"].append(epoch)
    ckan_history["loss"].append(np.mean(losses))
    ckan_history["eval_auc"].append(ev_auc)
    ckan_history["eval_f1"].append(ev_f1)
    ckan_history["test_auc"].append(te_auc)
    ckan_history["test_f1"].append(te_f1)

    print(f"{epoch:<6} | {np.mean(losses):<12.4f} | {ev_auc:<10.4f} {ev_f1:<10.4f} | {te_auc:<10.4f} {te_f1:<10.4f}")

    # Đóng gói và lưu checkpoint tốt nhất
    if te_auc > best_ckan_auc:
        best_ckan_auc = te_auc
        checkpoint_bundle = {
            "model_state_dict": ckan_model.state_dict(),
            "args": {
                "dim": DIM,
                "n_layer": N_LAYER,
                "agg": AGG,
                "batch_size": BATCH_SIZE,
                "use_cuda": torch.cuda.is_available()
            },
            "n_entity": n_entity,
            "n_relation": n_relation,
            "best_auc": best_ckan_auc,
            "best_f1": te_f1
        }
        torch.save(checkpoint_bundle, BEST_CKAN_PATH)

print("-" * 65)
print(f"[Hoàn tất] Huấn luyện CKAN! Peak Test AUC: {best_ckan_auc:.4f}")
print(f"Đã tự động lưu checkpoint xuất sắc nhất vào: {BEST_CKAN_PATH}")



## 8. Đánh Giá Top-K Recommendation (All-Ranking Protocol)
Đánh giá mô hình theo bài toán thực tế xếp hạng Top-K:
- Giao thức **All-ranking**: Chấm điểm toàn bộ các phim trong kho mà người dùng chưa từng tương tác trong tập train.
- Thang đo **Recall@K**: Đo lường tỷ lệ các bộ phim người dùng thực sự thích (Ground-truth trong tập test) xuất hiện trong Top-K gợi ý.
- Thang đo **NDCG@K (Normalized Discounted Cumulative Gain)**: Đánh giá chất lượng vị trí xếp hạng, cộng thưởng trọng số cao hơn khi phim yêu thích nằm ở đầu danh sách.


In [ ]:
# ============================================================
# 8. TOP-K RECOMMENDATION EVALUATION (RECALL@K, NDCG@K)
# ============================================================
def dcg_at_k(r, k):
    r = np.asarray(r, dtype=float)[:k]
    if r.size:
        return np.sum(r / np.log2(np.arange(2, r.size + 2)))
    return 0.0

def ndcg_at_k(r, k, ground_truth_count):
    dcg = dcg_at_k(r, k)
    ideal_r = [1] * min(k, ground_truth_count)
    idcg = dcg_at_k(ideal_r, k)
    return dcg / idcg if idcg > 0 else 0.0

def topk_eval(model, train_data, test_data, user_triple_set, item_triple_set, n_layer, n_item, device, k_list=[5, 10, 20], max_users=100, batch_size=2048):
    model.eval()
    user_train_pos = defaultdict(set)
    for u, i, r in train_data:
        if r == 1:
            user_train_pos[u].add(i)

    user_test_pos = defaultdict(set)
    for u, i, r in test_data:
        if r == 1:
            user_test_pos[u].add(i)

    common_users = list(set(user_train_pos.keys()) & set(user_test_pos.keys()))
    eval_users = np.random.choice(common_users, size=min(max_users, len(common_users)), replace=False)
    all_items = np.arange(n_item)

    recalls = {k: [] for k in k_list}
    ndcgs = {k: [] for k in k_list}

    print(f"Đang thực hiện Top-K Ranking trên {len(eval_users)} người dùng ngẫu nhiên với K={k_list}...")

    with torch.no_grad():
        for u in eval_users:
            seen = user_train_pos[u]
            ground_truth = user_test_pos[u]
            candidates = np.array([i for i in all_items if i not in seen])
            if len(candidates) == 0 or len(ground_truth) == 0:
                continue

            scores_list = []
            for c_start in range(0, len(candidates), batch_size):
                c_end = min(c_start + batch_size, len(candidates))
                c_batch = candidates[c_start:c_end]
                c_tensor = torch.LongTensor(c_batch).to(device)
                u_tr = to_triple_tensor([u] * len(c_batch), user_triple_set, n_layer, device)
                i_tr = to_triple_tensor(c_batch.tolist(), item_triple_set, n_layer, device)
                batch_scores = model(c_tensor, u_tr, i_tr)
                scores_list.extend(batch_scores.cpu().numpy().tolist())

            scores = np.array(scores_list)
            top_k_indices = np.argsort(scores)[::-1][:max(k_list)]
            top_items = candidates[top_k_indices]

            for k in k_list:
                k_items = top_items[:k]
                hits = [1 if item in ground_truth else 0 for item in k_items]
                recalls[k].append(sum(hits) / len(ground_truth))
                ndcgs[k].append(ndcg_at_k(hits, k, len(ground_truth)))

    results = {
        "Recall": {k: float(np.mean(recalls[k])) for k in k_list},
        "NDCG": {k: float(np.mean(ndcgs[k])) for k in k_list}
    }
    return results

# Tải checkpoint tốt nhất đã lưu để đánh giá Top-K
print("Đang nạp checkpoint tối ưu nhất ckan_model.pt để đánh giá Top-K...")
ckpt = torch.load(BEST_CKAN_PATH)
ckan_model.load_state_dict(ckpt["model_state_dict"])

topk_results = topk_eval(ckan_model, train_data, test_data, user_triple_set, item_triple_set, N_LAYER, n_item, device, k_list=[5, 10, 20])

print("-" * 55)
print(f"{'K':<6} | {'Recall@K':<15} | {'NDCG@K':<15}")
print("-" * 55)
for k in [5, 10, 20]:
    print(f"{k:<6} | {topk_results['Recall'][k]:<15.4f} | {topk_results['NDCG'][k]:<15.4f}")
print("-" * 55)



## 9. Trực Quan Hóa Đầy Đủ (CTR Prediction & Top-K Recommendation)
Vẽ biểu đồ đường biểu diễn sự hội tụ của mô hình và biểu đồ thanh đo lường Recall@K, NDCG@K.


In [ ]:
# ============================================================
# 9. TRỰC QUAN HÓA TOÀN DIỆN
# ============================================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Đường cong Loss
axes[0, 0].plot(ckan_history["epoch"], ckan_history["loss"], marker='o', color='#3b82f6', linewidth=2.5, label='CKAN Train Loss')
axes[0, 0].set_title("Hàm Mất Mát (Loss Curve)", fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("BCE Loss")
axes[0, 0].legend()

# 2. ROC-AUC & F1-Score (CTR Prediction)
axes[0, 1].plot(ckan_history["epoch"], ckan_history["test_auc"], marker='o', color='#059669', linewidth=2, label='Test AUC')
axes[0, 1].plot(ckan_history["epoch"], ckan_history["test_f1"], marker='s', color='#d97706', linewidth=2, label='Test F1')
axes[0, 1].set_title("CTR Prediction: ROC-AUC & F1-Score", fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Điểm số")
axes[0, 1].legend()

# 3. Top-K Recall@K
k_labels = [f"K={k}" for k in [5, 10, 20]]
recall_vals = [topk_results["Recall"][k] for k in [5, 10, 20]]
bars_r = axes[1, 0].bar(k_labels, recall_vals, color=['#6ee7b7', '#34d399', '#059669'], width=0.5)
axes[1, 0].set_title("Top-K Recommendation: Recall@K", fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel("Recall")
for p in bars_r:
    axes[1, 0].annotate(f"{p.get_height():.4f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 3),
                        textcoords='offset points')

# 4. Top-K NDCG@K
ndcg_vals = [topk_results["NDCG"][k] for k in [5, 10, 20]]
bars_n = axes[1, 1].bar(k_labels, ndcg_vals, color=['#93c5fd', '#60a5fa', '#2563eb'], width=0.5)
axes[1, 1].set_title("Top-K Recommendation: NDCG@K", fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel("NDCG")
for p in bars_n:
    axes[1, 1].annotate(f"{p.get_height():.4f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 3),
                        textcoords='offset points')

plt.tight_layout()
plt.show()

print("\n--- BẢNG TỔNG KẾT TẤT CẢ THANG ĐO ---")
print(f"1. CTR PREDICTION:")
print(f"   - Peak Test ROC-AUC : {max(ckan_history['test_auc']):.4f}")
print(f"   - Peak Test F1-Score: {max(ckan_history['test_f1']):.4f}")
print(f"2. TOP-K RECOMMENDATION:")
for k in [5, 10, 20]:
    print(f"   - K={k:2d}: Recall@{k} = {topk_results['Recall'][k]:.4f} | NDCG@{k} = {topk_results['NDCG'][k]:.4f}")



## 10. Tải Mô Hình CKAN Đã Huấn Luyện Về Máy Tính
Chạy ô code dưới đây để kích hoạt trình duyệt tự động tải file `ckan_model.pt` về máy tính:


In [ ]:
# ============================================================
# 10. TẢI FILE MÔ HÌNH VỀ MÁY TÍNH
# ============================================================
from google.colab import files

MODEL_PATH = "./models/ckan_model.pt"

if os.path.exists(MODEL_PATH):
    file_size_mb = os.path.getsize(MODEL_PATH) / 1e6
    print(f"Đang tải file mô hình {MODEL_PATH} ({file_size_mb:.2f} MB) về máy tính...")
    files.download(MODEL_PATH)
    print("[OK] Đã kích hoạt tải về trên trình duyệt!")
else:
    print("[Lỗi] Không tìm thấy file checkpoint. Vui lòng chạy bước huấn luyện CKAN ở mục 7 trước.")



### Hướng Dẫn Sử Dụng Mô Hình Trong Dự Án Local
1. Đặt file `ckan_model.pt` vừa tải về vào thư mục backend của dự án:
   ```bash
   dss_ckan_movie_recommender_system/backend/models/ckan_model.pt
   ```
2. Khởi động lại Backend FastAPI:
   ```bash
   uv run uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
   ```
3. Backend sẽ tự động phát hiện và nạp trọng số mô hình:
   ```log
   INFO: Loading CKAN checkpoint from backend/models/ckan_model.pt...
   INFO: CKAN checkpoint loaded successfully.
   ```
Toàn bộ hệ thống gợi ý và đồ thị tri thức trên frontend `http://localhost:5173` sẽ lập tức sử dụng mô hình vừa huấn luyện!
